# Experimento 05 — Análise de Domain Gap: MEGNet Bulk → 2D

## Hipótese

> O MEGNet treinado em bulk (fine-tuning MP→C2DB) apresenta erro maior do que o MEGNet
> scratch no domínio 2D, e esse erro se concentra nos materiais com maior Δgap (bulk→2D).

## Objetivo

Reutilizar os checkpoints de `run/004` (scratch e fine-tuning), reconstruir o mesmo split
de teste (seed=42) e cruzar os erros de predição com o Δgap do dataset pareado (`run/006`).

## Cenários

| Cenário | Modelo           | Checkpoint           |
|---------|------------------|----------------------|
| A       | MEGNet scratch   | run/004/model/scratch |
| B       | MEGNet fine-tune | run/004/model/finetune|

## Referências

- Chen et al. 2019 — MEGNet (`1812.05055v2.pdf`)
- Ko et al. 2025 — MatGL (`s41524-025-01742-y-1.pdf`)
- Gjerding et al. 2021 — C2DB

In [ ]:
from __future__ import annotations

import warnings
from copy import deepcopy
from functools import partial
from pathlib import Path

import ase.db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Subset, DataLoader

from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.data import MGLDataset, collate_fn_graph
from matgl.models import MEGNet
from matgl.utils.training import ModelLightningModule

import os
NOTEBOOK_DIR = Path(os.path.abspath(''))

warnings.simplefilter("ignore")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

In [ ]:
# ── Identificação do Run ─────────────────────────────────────────
RUN_ID   = "003"
RUN_NAME = "domain_gap"
RUN_DIR = ensure_run_dir(RUN_ID, RUN_NAME)

RUN_001 = RUNS_DIR / "001_megnet_finetune_c2db"
RUN_002 = RUNS_DIR / "002_bulk2d_analysis"

CKPT_SCRATCH  = sorted((RUN_001 / "model" / "scratch").glob("*.ckpt"))[-1]
CKPT_FINETUNE = sorted((RUN_001 / "model" / "finetune").glob("*.ckpt"))[-1]
print(f"Scratch ckpt : {CKPT_SCRATCH.name}")
print(f"Finetune ckpt: {CKPT_FINETUNE.name}")

PAIRED_CSV = RUN_002 / "outputs" / "paired_pbe.csv"
assert PAIRED_CSV.exists(), f"Execute o Exp 002 primeiro: {PAIRED_CSV}"

SEED        = 42
EHULL_TRAIN = 0.2
CUTOFF      = 5.0

def _safe_device() -> str:
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        torch.zeros(1).cuda()
        return 'cuda'
    except Exception:
        return 'cpu'

DEVICE = _safe_device()
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 1. Reconstruir Dataset C2DB (mesmo split de run/004)

Replica exatamente o carregamento do run/004:
- mesmo `ehull ≤ 0.2`
- mesma ordem de iteração do C2DB SQLite
- mesmo `SEED=42` para `np.random.default_rng`
- resultado: mesmo `test_idx` ← necessário para análise de erro

In [ ]:
C2DB_PATH = required_path(DATA_DIR / "raw" / "c2db.db", "C2DB")
adaptor   = AseAtomsAdaptor()


def load_c2db(db_path, max_ehull=0.2):
    db = ase.db.connect(str(db_path))
    structures, gap_pbe, gap_hse = {}, {}, {}
    for row in db.select():
        uid = row.get('uid') or str(row.id)
        ehull = row.get('ehull')
        if ehull is not None and ehull > max_ehull:
            continue
        g_pbe = row.get('gap')
        g_hse = row.get('gap_hse')
        if g_pbe is None and g_hse is None:
            continue
        try:
            struct = adaptor.get_structure(row.toatoms())
        except Exception:
            continue
        structures[uid] = struct
        gap_pbe[uid]    = g_pbe
        gap_hse[uid]    = g_hse
    return structures, gap_pbe, gap_hse


structures, gap_pbe, gap_hse = load_c2db(C2DB_PATH, max_ehull=EHULL_TRAIN)
print(f"C2DB carregado: {len(structures)} materiais")

In [ ]:
# Montar listas (mesma lógica de run/004 — PBE primeiro, depois HSE)
all_structures = []
all_fidelities = []
all_targets    = []
all_uids       = []

for uid, struct in structures.items():
    g_pbe = gap_pbe.get(uid)
    if g_pbe is not None and g_pbe > 0:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(0)
        all_targets.append(float(g_pbe))
        all_uids.append(f'{uid}_pbe')

    g_hse = gap_hse.get(uid)
    if g_hse is not None:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(1)
        all_targets.append(float(g_hse))
        all_uids.append(f'{uid}_hse')

print(f"Total de amostras: {len(all_structures)}")
print(f"  PBE: {all_fidelities.count(0)}")
print(f"  HSE: {all_fidelities.count(1)}")

In [ ]:
# MGLDataset — reutiliza cache de run/005 se disponível
CACHE_DIR = str(RUN_DIR / "cache" / "MGLDataset")
CACHE_DIR_ABS = str(Path(CACHE_DIR).resolve())

element_types = DEFAULT_ELEMENTS
cry_graph     = Structure2Graph(element_types=element_types, cutoff=CUTOFF)

dataset = MGLDataset(
    structures=all_structures,
    graph_labels=all_fidelities,
    labels={"bandgap": all_targets},
    converter=cry_graph,
    filename="C2DB_MGLDataset_exp05",
)
print(f"Dataset: {len(dataset)} amostras")

In [ ]:
# Reproduzir o mesmo split de run/004
n = len(all_uids)
c2db_idx = list(range(n))  # todos são C2DB (sem JARVIS aqui)

rng = np.random.default_rng(SEED)
shuffled = rng.permutation(c2db_idx).tolist()

n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_idx = shuffled[:n_train]
val_idx   = shuffled[n_train:n_train + n_val]
test_idx  = shuffled[n_train + n_val:]

test_data = Subset(dataset, test_idx)
print(f"Test set: {len(test_data)} amostras")

collate_fn = partial(collate_fn_graph, include_line_graph=False)
test_loader = DataLoader(
    test_data, batch_size=256, shuffle=False,
    collate_fn=collate_fn, num_workers=4,
)

## 2. Carregar Modelos e Rodar Inferência

In [ ]:
def build_megnet_c2db(dim_state=64):
    return MEGNet(
        element_types=DEFAULT_ELEMENTS,
        cutoff=CUTOFF,
        is_intensive=True,
        dim_state_embedding=dim_state,
        ntypes_state=2,
        readout_type='set2set',
        include_states=True,
    )


def load_megnet_ckpt(ckpt_path: Path) -> MEGNet:
    model = build_megnet_c2db()
    lit   = ModelLightningModule.load_from_checkpoint(
        str(ckpt_path), model=model, map_location='cpu'
    )
    return lit.model.to(DEVICE).eval()


model_scratch  = load_megnet_ckpt(CKPT_SCRATCH)
model_finetune = load_megnet_ckpt(CKPT_FINETUNE)
print("Modelos carregados.")

In [ ]:
@torch.no_grad()
def run_inference(model, loader, uids_all, idx_list):
    preds, reals, fids = [], [], []
    for batch in loader:
        graphs, lat, state, labels = batch
        graphs = graphs.to(DEVICE)
        lat    = lat.to(DEVICE)
        state  = state.to(DEVICE) if state is not None else state

        graphs.edata["lattice"]      = torch.repeat_interleave(lat, graphs.batch_num_edges(), dim=0)
        graphs.edata["pbc_offshift"] = (graphs.edata["pbc_offset"].unsqueeze(-1) * graphs.edata["lattice"]).sum(dim=1)
        graphs.ndata["pos"]          = (graphs.ndata["frac_coords"].unsqueeze(-1) * torch.repeat_interleave(lat, graphs.batch_num_nodes(), dim=0)).sum(dim=1)

        out = model(graphs, state)
        preds.extend(out.cpu().numpy().flatten().tolist())

        # labels pode ser dict {"bandgap": tensor} ou tensor direto
        target = labels['bandgap'] if isinstance(labels, dict) else labels
        reals.extend(target.numpy().flatten().tolist())

    uids_test = [uids_all[i] for i in idx_list]
    fids_test = [all_fidelities[i] for i in idx_list]

    df = pd.DataFrame({
        'uid'      : uids_test,
        'fidelity' : fids_test,
        'real'     : reals,
        'pred'     : preds,
        'abs_error': np.abs(np.array(reals) - np.array(preds)),
        'bias'     : np.array(preds) - np.array(reals),
    })
    df['uid_base'] = df['uid'].str.rsplit('_', n=1).str[0]
    return df


df_scratch  = run_inference(model_scratch,  test_loader, all_uids, test_idx)
df_finetune = run_inference(model_finetune, test_loader, all_uids, test_idx)

print(f"Resultados scratch : {len(df_scratch)} amostras")
print(f"Resultados finetune: {len(df_finetune)} amostras")


## 3. Métricas Globais

In [ ]:
def compute_metrics(df, name=''):
    mae    = df['abs_error'].mean()
    rmse   = np.sqrt((df['bias']**2).mean())
    bias   = df['bias'].mean()
    # UWBG subset (real > 3.4 eV)
    uwbg   = df[df['real'] > 3.4]
    mae_u  = uwbg['abs_error'].mean() if len(uwbg) > 0 else np.nan
    bias_u = uwbg['bias'].mean()      if len(uwbg) > 0 else np.nan
    # HSE only
    hse    = df[df['fidelity'] == 1]
    mae_h  = hse['abs_error'].mean()  if len(hse) > 0 else np.nan
    return {
        'model'   : name,
        'MAE'     : round(mae, 4),
        'RMSE'    : round(rmse, 4),
        'Bias'    : round(bias, 4),
        'MAE_HSE' : round(mae_h, 4),
        'MAE_UWBG': round(mae_u, 4),
        'Bias_UWBG': round(bias_u, 4),
        'n'       : len(df),
        'n_UWBG'  : len(uwbg),
    }


metrics = pd.DataFrame([
    compute_metrics(df_scratch,  'MEGNet Scratch (A)'),
    compute_metrics(df_finetune, 'MEGNet Fine-tune (B)'),
])
print(metrics.to_string(index=False))

In [ ]:
# ── Scatter predito vs real (ambos os modelos) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
UWBG_TH = 3.4

for ax, (df, title) in zip(axes, [
    (df_scratch, 'MEGNet Scratch (A)'),
    (df_finetune, 'MEGNet Fine-tune (B)'),
]):
    hse_mask = df['fidelity'] == 1
    ax.scatter(df.loc[~hse_mask, 'real'], df.loc[~hse_mask, 'pred'],
               alpha=0.2, s=5, color='steelblue', label='PBE', rasterized=True)
    ax.scatter(df.loc[hse_mask, 'real'],  df.loc[hse_mask, 'pred'],
               alpha=0.5, s=8, color='darkorange', label='HSE', rasterized=True)
    lim = max(df['real'].max(), df['pred'].max()) + 0.5
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
    ax.axvline(UWBG_TH, color='red', linestyle=':', linewidth=0.8, alpha=0.7)
    ax.axhline(UWBG_TH, color='red', linestyle=':', linewidth=0.8, alpha=0.7)
    m = compute_metrics(df)
    ax.set_title(f"{title}\nMAE={m['MAE']:.3f} eV | RMSE={m['RMSE']:.3f} eV", fontsize=11)
    ax.set_xlabel('Gap real (eV)')
    ax.set_ylabel('Gap predito (eV)')
    ax.legend(fontsize=8, markerscale=2)

fig.suptitle('Scatter: Predito vs Real — Cenários A e B', fontsize=13)
plt.tight_layout()
plt.savefig(RUN_DIR / 'figures' / 'scatter_scratch_vs_finetune.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análise de Erro vs Δgap (Dataset Pareado)

Cruza os erros de predição no conjunto de teste com o Δgap (bulk→2D) do Exp 04.
Materiais do teste que aparecem no dataset pareado permitem verificar se o erro aumenta
nos materiais com maior variação de gap na transição bulk→2D.

In [ ]:
df_paired = pd.read_csv(PAIRED_CSV)
print(f"Dataset pareado carregado: {len(df_paired)} pares")
print(df_paired[['uid','reduced_formula','gap_pbe_2d','gap_pbe_bulk_mean','delta_gap_pbe']].head())

In [ ]:
def merge_with_paired(df_pred, df_paired, fidelity=1):
    """
    Cruza predições do modelo (fidelidade HSE ou PBE) com dataset pareado.
    """
    # Filtra pela fidelidade desejada
    pred_fid = df_pred[df_pred['fidelity'] == fidelity].copy()
    pred_fid['uid_base'] = pred_fid['uid_base']  # já existe

    # Merge por uid_base = uid do C2DB
    merged = pred_fid.merge(
        df_paired[['uid','reduced_formula','gap_pbe_2d','gap_pbe_bulk_mean','delta_gap_pbe',
                   'gap_hse_2d','gap_hse_bulk_mean','delta_gap_hse']],
        left_on='uid_base', right_on='uid', how='inner',
    )
    print(f"  Pares encontrados no test set (fid={fidelity}): {len(merged)}")
    return merged


print("Scratch:")
merged_scratch  = merge_with_paired(df_scratch,  df_paired, fidelity=1)
print("Fine-tune:")
merged_finetune = merge_with_paired(df_finetune, df_paired, fidelity=1)

In [ ]:
# ── Erro absoluto vs Δgap ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (merged, title, color) in zip(axes, [
    (merged_scratch,  'MEGNet Scratch (A)', 'steelblue'),
    (merged_finetune, 'MEGNet Fine-tune (B)', 'darkorange'),
]):
    if len(merged) < 5:
        ax.text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center',
                transform=ax.transAxes)
        continue

    ax.scatter(merged['delta_gap_pbe'], merged['abs_error'],
               alpha=0.5, s=20, color=color, rasterized=True)

    # Tendência (regressão linear)
    mask = merged['delta_gap_pbe'].notna() & merged['abs_error'].notna()
    if mask.sum() > 10:
        coeffs = np.polyfit(merged.loc[mask,'delta_gap_pbe'], merged.loc[mask,'abs_error'], 1)
        xp = np.linspace(merged['delta_gap_pbe'].min(), merged['delta_gap_pbe'].max(), 100)
        ax.plot(xp, np.polyval(coeffs, xp), 'r-', linewidth=1.5,
                label=f'slope={coeffs[0]:.3f}')

    corr = merged[['delta_gap_pbe','abs_error']].corr().iloc[0,1]
    ax.set_xlabel('Δgap PBE (bulk → 2D) [eV]')
    ax.set_ylabel('Erro absoluto HSE (eV)')
    ax.set_title(f'{title}\nr={corr:.3f}')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend(fontsize=9)

fig.suptitle('Erro Absoluto vs Δgap Bulk→2D (subset pareado, HSE)', fontsize=13)
plt.tight_layout()
plt.savefig(RUN_DIR / 'figures' / 'error_vs_delta_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Viés vs Δgap ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (merged, title, color) in zip(axes, [
    (merged_scratch,  'MEGNet Scratch (A)', 'steelblue'),
    (merged_finetune, 'MEGNet Fine-tune (B)', 'darkorange'),
]):
    if len(merged) < 5:
        ax.text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center',
                transform=ax.transAxes)
        continue

    ax.scatter(merged['delta_gap_pbe'], merged['bias'],
               alpha=0.5, s=20, color=color, rasterized=True)

    mask = merged['delta_gap_pbe'].notna() & merged['bias'].notna()
    if mask.sum() > 10:
        coeffs = np.polyfit(merged.loc[mask,'delta_gap_pbe'], merged.loc[mask,'bias'], 1)
        xp = np.linspace(merged['delta_gap_pbe'].min(), merged['delta_gap_pbe'].max(), 100)
        ax.plot(xp, np.polyval(coeffs, xp), 'r-', linewidth=1.5,
                label=f'slope={coeffs[0]:.3f}')

    corr = merged[['delta_gap_pbe','bias']].corr().iloc[0,1]
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Δgap PBE (bulk → 2D) [eV]')
    ax.set_ylabel('Viés (pred − real) [eV]')
    ax.set_title(f'{title}\nr={corr:.3f}')
    ax.legend(fontsize=9)

fig.suptitle('Viés vs Δgap Bulk→2D (subset pareado, HSE)', fontsize=13)
plt.tight_layout()
plt.savefig(RUN_DIR / 'figures' / 'bias_vs_delta_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Estratificação por faixas de Δgap ────────────────────────────
def stratify_by_delta_gap(merged, label):
    """Computa MAE por faixa de Δgap."""
    bins   = [-np.inf, -1, 0, 1, 2, np.inf]
    labels = ['< -1', '-1 a 0', '0 a 1', '1 a 2', '> 2']
    merged = merged.copy()
    merged['delta_bin'] = pd.cut(merged['delta_gap_pbe'], bins=bins, labels=labels)
    tab = merged.groupby('delta_bin', observed=True).agg(
        n=('abs_error', 'count'),
        mae=('abs_error', 'mean'),
        rmse=('bias', lambda x: np.sqrt((x**2).mean())),
        bias=('bias', 'mean'),
    ).round(3)
    tab.index.name = f'Δgap bin [{label}]'
    return tab


print("── Scratch (A) ──")
tab_s = stratify_by_delta_gap(merged_scratch, 'Scratch')
print(tab_s.to_string())

print("\n── Fine-tune (B) ──")
tab_f = stratify_by_delta_gap(merged_finetune, 'Fine-tune')
print(tab_f.to_string())

## 5. Materiais Onde Fine-tuning Falha Mais

In [ ]:
if len(merged_finetune) > 0 and len(merged_scratch) > 0:
    # Merge dos dois para comparação direta
    comp = merged_scratch[['uid_base','reduced_formula','real','abs_error','delta_gap_pbe']].copy()
    comp.rename(columns={'abs_error':'err_scratch'}, inplace=True)

    ft = merged_finetune[['uid_base','abs_error']].rename(columns={'abs_error':'err_finetune'})
    comp = comp.merge(ft, on='uid_base', how='inner')
    comp['delta_err'] = comp['err_finetune'] - comp['err_scratch']  # + = finetune pior

    print("Top 15 materiais onde fine-tune é pior que scratch (HSE):")
    print(
        comp.nlargest(15, 'delta_err')
        [['reduced_formula','real','err_scratch','err_finetune','delta_err','delta_gap_pbe']]
        .round(3).to_string(index=False)
    )

## 6. Salvar Resultados

In [ ]:
metrics.to_csv(RUN_DIR / 'outputs' / 'metrics_comparison.csv', index=False)
df_scratch.to_csv(RUN_DIR / 'outputs' / 'predictions_scratch.csv', index=False)
df_finetune.to_csv(RUN_DIR / 'outputs' / 'predictions_finetune.csv', index=False)

if len(merged_scratch) > 0:
    merged_scratch.to_csv(RUN_DIR / 'outputs' / 'paired_errors_scratch.csv', index=False)
if len(merged_finetune) > 0:
    merged_finetune.to_csv(RUN_DIR / 'outputs' / 'paired_errors_finetune.csv', index=False)

print("Outputs salvos em:", RUN_DIR / 'outputs')

In [ ]:
print("=" * 60)
print("RESUMO — Experimento 05: MEGNet Domain Analysis")
print("=" * 60)
print(metrics.to_string(index=False))
print()
scratch_better = metrics.loc[metrics['model'].str.contains('Scratch'), 'MAE'].values[0] < \
                 metrics.loc[metrics['model'].str.contains('Fine'), 'MAE'].values[0]
print("Scratch supera fine-tune em MAE global:", scratch_better)

if len(merged_scratch) > 5:
    corr_s = merged_scratch[['delta_gap_pbe','abs_error']].corr().iloc[0,1]
    corr_f = merged_finetune[['delta_gap_pbe','abs_error']].corr().iloc[0,1] if len(merged_finetune) > 5 else np.nan
    print(f"Correlação erro vs Δgap — Scratch: {corr_s:.3f} | Fine-tune: {corr_f:.3f}")
print("=" * 60)

In [ ]:
import shutil

nb_src = NOTEBOOK_DIR / "megnet_domain_analysis.ipynb"
nb_dst = RUN_DIR / "notebook.ipynb"
if nb_src.exists():
    shutil.copy2(nb_src, nb_dst)
    print(f"Notebook salvo em: {nb_dst.resolve()}")